In [1]:
!pip install pyspark

import os
import shutil

# 1. Data Lake Structure
os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver/preprocessed_sales', exist_ok=True)
os.makedirs('silver', exist_ok=True) # For final github submission

# Move uploaded file to Bronze layer
if os.path.exists('sales.csv'):
    shutil.copy('sales.csv', 'data_lake/bronze/sales.csv')
    print("Folders created successfully.")
    print("sales.csv moved to the Bronze layer.")
else:
    print("WARNING: 'sales.csv' not found. Please upload it to Colab first.")

Folders created successfully.
sales.csv moved to the Bronze layer.


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, to_date, when, count

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("DataLakePreprocessing") \
    .getOrCreate()

# 2. Load and Inspect
bronze_path = "data_lake/bronze/sales.csv"
df_bronze = spark.read.csv(bronze_path, header=True, inferSchema=True)

print("--- Bronze Layer Schema ---")
df_bronze.printSchema()

print("--- Bronze Layer Data Sample ---")
df_bronze.show(5, truncate=False)

initial_count = df_bronze.count()
print(f"Total Initial Records: {initial_count}")

print("\n--- Identifying Exact Duplicates ---")
duplicates = initial_count - df_bronze.dropDuplicates().count()
print(f"Found {duplicates} exact duplicate rows.")

print("\n--- Identifying Null Values by Column ---")
# UPDATED: Removed isnan() to prevent string casting errors
df_bronze.select([count(when(col(c).isNull(), c)).alias(c) for c in df_bronze.columns]).show()

--- Bronze Layer Schema ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

--- Bronze Layer Data Sample ---
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+-

In [8]:
from pyspark.sql.functions import col, trim, initcap, udf
from pyspark.sql.types import DateType
from datetime import datetime
import glob
import shutil

# 3. Basic Preprocessing - Silver Layer

# Requirement 1: Remove exact duplicates
df_silver = df_bronze.dropDuplicates()

# Requirement 2: Handle missing values
df_silver = df_silver.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

# Requirement 3: Trim spaces and standardize capitalization
text_columns = ["customer_id", "customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
for c in text_columns:
    df_silver = df_silver.withColumn(c, trim(initcap(col(c))))

# Requirement 4: Basic validation (Keep valid quantities, prices, and discounts)
df_silver = df_silver.filter(
    (col("quantity") > 0) &
    (col("unit_price") > 0) &
    (col("discount_percent") >= 0) &
    (col("discount_percent") <= 100)
)

# Requirement 5: Convert valid dates to one format and handle invalid ones
# Using a PySpark UDF to safely parse dates and avoid Spark strict mode crashes
def safe_parse_date(date_str):
    if not date_str:
        return None
    # Add any date formats present in the dataset here
    formats_to_try = ["%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y"]
    for fmt in formats_to_try:
        try:
            # If the date is impossible (e.g., Feb 31), ValueError is caught here
            return datetime.strptime(date_str, fmt).date()
        except ValueError:
            continue
    return None # Returns None (NULL in Spark) if it is entirely invalid

parse_date_udf = udf(safe_parse_date, DateType())

# Apply the UDF to the order_date column
df_silver = df_silver.withColumn("order_date", parse_date_udf(col("order_date")))

# Filter out the invalid dates (which the UDF converted to NULL)
df_silver = df_silver.filter(col("order_date").isNotNull())

# Requirement 6: Preserved useful records without over-cleaning

# Save the resulting DataFrame
silver_lake_path = "data_lake/silver/preprocessed_sales"
df_silver.coalesce(1).write.mode("overwrite").csv(silver_lake_path, header=True)

# Format final file for GitHub upload
csv_files = glob.glob(f"{silver_lake_path}/*.csv")
if csv_files:
    shutil.copy(csv_files[0], "silver/sales_silver.csv")
    print("Preprocessed data saved. 'silver/sales_silver.csv' is ready for download.")

Preprocessed data saved. 'silver/sales_silver.csv' is ready for download.


In [9]:
# 4. Verification
final_count = df_silver.count()
print(f"Row count before preprocessing: {initial_count}")
print(f"Row count after preprocessing:  {final_count}")

print("\n--- Verifying Silver Layer Data ---")
df_silver_read = spark.read.csv("silver/sales_silver.csv", header=True, inferSchema=True)
df_silver_read.show(5, truncate=False)

# Brief statement of operations
print("""
--- Preprocessing Operations Performed ---
1. Removed all exact duplicate rows.
2. Filled missing values in text columns (e.g., customer_name) with 'Unknown'.
3. Trimmed extra whitespaces and applied Title Case to text fields for standardization.
4. Applied numerical validation: Dropped rows with quantity <= 0, unit_price <= 0, or discount outside 0-100%.
5. Cast 'order_date' to PySpark DateType (YYYY-MM-DD) and filtered out unparseable dates.
""")

Row count before preprocessing: 1000
Row count after preprocessing:  957

--- Verifying Silver Layer Data ---
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |